# Гир 1.5: канонический score волатильности (heatmap + Top‑10)

Интерактивный разбор **одного** зафиксированного screener-score для отбора горячего кластера монет.

---

## Зачем нужна метрика

Метрика нужна, чтобы на срезе рынка в момент `t` **найти Top‑N / кластер «горячих» монет** — кандидатов для дальнейшего разбора и (позже) очереди капитала гира 2.

Она **не** отвечает на вопрос «монета A строго волатильнее монеты B в попарном смысле».  
Разница score между соседними рангами — это **ранжирование внутри среза**, а не физическая шкала «насколько волатильнее».

Одновременно **интегрально по рынку** тот же score несёт информацию о **рыночной волатильности в точке времени**: средняя по панели / нижней полосе heatmap («средняя crypto») показывает, насколько рынок в целом «горяч» относительно собственного фона.

---

## Каноническая формула (зафиксирована)

Короткий blend EMA и MA в числителе, длинная SMA в знаменателе; затем geom (или другой `COMPOSITE_VARIANT`) по volume и ATR:

```text
r_* = (α · EMA_short + (1 − α) · MA_short) / MA_long

composite = √(r_vol · r_atr)     # default: geom
```

| Параметр | Default | Смысл |
|----------|---------|--------|
| `α` (`BLEND_ALPHA`) | **0.75** | soft short blend: больше веса у EMA, MA смягчает шум |
| `MA_SHORT` | `6` | ~30 мин на барах 5m |
| `MA_LONG` | `288` | ~1 сутки фона (как в вашем окне) |
| `COMPOSITE_VARIANT` | `geom` | равный log-вес volume и ATR |

Код: [`research/regime_ma_ratio.py`](regime_ma_ratio.py) (`numerator="blend"`).  
Документ: [`docs/regime-metrics-v0.md`](../docs/regime-metrics-v0.md).

> **Замечание по классам активов.** Дефолт ноутбука — `ASSET_CLASS = "crypto"`.  
> Для equity / ETF та же метрика **требует отдельной доработки**: сильная временная неоднородность объёма и амплитуды (сессии, аукционы, overnight) ломает сопоставимость short/long окон без доп. нормализации.

---

## Что осталось в ноутбуке

1. Imports  
2. **Один CONFIG**  
3. Загрузка баров + features (blend)  
4. Панель + heatmap (+ средняя по рынку)  
5. **Top‑10** на выбранном timestamp (таблица + detail-графики)

Всё остальное (сравнение MA/EMA/ema_pct_z, Top‑3 dual CONFIG, переходы top‑1, corr-ячейки) убрано — один metric path.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

import importlib

REPO = Path("..").resolve()
if not (REPO / "research" / "regime_ma_ratio.py").exists():
    REPO = Path.cwd().resolve()
sys.path.insert(0, str(REPO))

import research.regime_metrics as _rm
import research.regime_ma_ratio as _mr
import research.rank_volatile_coins as _rvc
import research.regime_composite as _rc

importlib.reload(_rm)
importlib.reload(_mr)
importlib.reload(_rc)
importlib.reload(_rvc)

from research.regime_ma_ratio import (
    BLEND_ALPHA_DEFAULT,
    NUMERATOR_BLEND,
    SCORE_MODE_MA_RATIO,
    VARIANTS,
    MaRatioParams,
    build_ma_ratio_features,
    build_ma_ratio_panel,
    formula_doc,
    numerator_formula,
    select_composite_column,
)
from research.regime_metrics import RegimeParams
from research.regime_composite import merge_exchange_composites
from research.rank_volatile_coins import (
    BYBIT_ROOT,
    OKX_ROOT,
    WARMUP_BARS_PAD,
    format_ts,
    list_coins,
    load_hist_bars_recent,
    warmup_min_ts,
)
from research.is_crypto import filter_crypto_coins, is_crypto

print("REPO", REPO)
print("okx", OKX_ROOT.exists(), "bybit", BYBIT_ROOT.exists())
print("canonical numerator", NUMERATOR_BLEND, "α default", BLEND_ALPHA_DEFAULT)


## CONFIG

Один блок параметров. После смены knobs:

- только `TOP10_*` / heatmap display → CONFIG → heatmap и/или Top‑10  
- `START`/`END`/`MAX_COINS`/`ASSET_CLASS`/`MA_*`/`BLEND_ALPHA`/биржи → CONFIG → загрузка → панель → heatmap → Top‑10

`ASSET_CLASS="crypto"` — дефолт. Equity оставляем в API фильтра, но метрика для акций **не канон** (см. шапку).


In [ ]:
# --- данные ---
EXCHANGES = ["okx"]  # "okx" | "bybit" | оба: ["okx", "bybit"]
COMBINE = "mean"  # mean | min — только если обе биржи
OKX_DATA = OKX_ROOT
BYBIT_DATA = BYBIT_ROOT

# календарное окно панели (UTC). END exclusive.
# Полный месяц дампа: START="2026-07-08", END="2026-08-08"
START = "2026-08-01"
END = "2026-08-08"

MAX_COINS = 0  # 0 = все; алфавитный срез если нет allowlist
COIN_ALLOWLIST: list[str] | None = None  # напр. ["BTC", "ETH", "SOL"]

# --- класс активов ---
# Default crypto. Equity: та же формула экспериментальна (сессии / объём неоднородны).
ASSET_CLASS = "crypto"  # "crypto" | "equity" | "both"
ASSET_CLASS_ALLOWED = ("crypto", "equity", "both")
if ASSET_CLASS not in ASSET_CLASS_ALLOWED:
    raise ValueError(f"ASSET_CLASS must be one of {ASSET_CLASS_ALLOWED}, got {ASSET_CLASS!r}")

# --- канонический score: blend short / MA long ---
SCORE_MODE = SCORE_MODE_MA_RATIO  # зафиксировано: ma_ratio
COMPOSITE_VARIANT = "geom"  # geom | mean | min | log_mean | vol_only | atr_only
MA_SHORT = 6
MA_LONG = 288  # ~1 day of 5m bars
MA_EXTRA_LONGS: tuple[int, ...] = ()  # не нужно для heatmap/Top-10
ATR_N = 1  # 1 = true range; >1 = SMA(TR) before ratio MAs
NUMERATOR = NUMERATOR_BLEND  # canonical
BLEND_ALPHA = BLEND_ALPHA_DEFAULT  # ≈ 0.75 soft short blend

MA_PARAMS = MaRatioParams(
    short=MA_SHORT,
    long=MA_LONG,
    atr_n=ATR_N,
    numerator=NUMERATOR,
    blend_alpha=BLEND_ALPHA,
    regime=RegimeParams(),
)

# --- панель / heatmap ---
PANEL_STRIDE = 1  # 1 = каждый 5m бар; >1 = preview
COIN_SORT = "mean"  # mean | max | alpha
HEATMAP_TIME_DOWNSAMPLE = 1  # только display
COLORSCALE = "Viridis"
STORE_ALL_VARIANTS = False  # один metric path — лишние composite_* не копим

# --- Top-10 @ timestamp ---
TOP10_TS = "2026-08-06T15:00:00Z"  # ISO UTC или int ms
TOP10_WINDOW_HOURS = 6  # ± окно detail-графиков вокруг TOP10_TS
TOP_N = 10
PLOT_TOP10_DETAILS = True  # False → только таблица

if COMPOSITE_VARIANT not in VARIANTS:
    raise ValueError(f"COMPOSITE_VARIANT must be one of {VARIANTS}")

SCORE_LABEL = (
    f"{COMPOSITE_VARIANT}: {formula_doc(COMPOSITE_VARIANT)}  "
    f"{MA_PARAMS.numerator_label}  "
    f"({numerator_formula(NUMERATOR, blend_alpha=BLEND_ALPHA)})"
)
ACTIVE_PARAMS = MA_PARAMS

print("window", START, "→", END, "(END exclusive)")
print("exchanges", EXCHANGES, "combine", COMBINE if len(EXCHANGES) > 1 else None)
print("asset_class", ASSET_CLASS, "| filter via research.is_crypto")
print("max_coins", MAX_COINS, "allowlist", COIN_ALLOWLIST)
print("score", SCORE_LABEL)
print("ma", {
    "short": MA_SHORT,
    "long": MA_LONG,
    "atr_n": ATR_N,
    "numerator": NUMERATOR,
    "blend_alpha": BLEND_ALPHA,
})
print("stride", PANEL_STRIDE, "coin_sort", COIN_SORT, "hm_downsample", HEATMAP_TIME_DOWNSAMPLE)
print("top10", "@", TOP10_TS, f"±{TOP10_WINDOW_HOURS}h", "TOP_N", TOP_N, "plots", PLOT_TOP10_DETAILS)


## Загрузка баров + causal features

На каждую монету: hist `5m` с warmup до `START`. Features — канонический blend (`r_vol`, `r_atr`, `composite_<variant>`).


In [ ]:
BAR_MS = 300_000


def day_to_ms(day: str) -> int:
    dt = datetime.fromisoformat(day).replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)


start_ms = day_to_ms(START)
end_ms = day_to_ms(END) - BAR_MS  # last bar whose start is still < END day
if end_ms < start_ms:
    raise ValueError("END must be after START")

load_min_ms = warmup_min_ts(end_ms, ACTIVE_PARAMS, score_mode=SCORE_MODE)
load_min_ms = min(load_min_ms, warmup_min_ts(start_ms, ACTIVE_PARAMS, score_mode=SCORE_MODE))
if MA_EXTRA_LONGS:
    max_long = max(MA_LONG, max(MA_EXTRA_LONGS))
else:
    max_long = MA_LONG
pad = (max_long + ATR_N + MA_SHORT + WARMUP_BARS_PAD) * BAR_MS
load_min_ms = min(load_min_ms, end_ms - pad, start_ms - pad)

primary = "okx" if "okx" in EXCHANGES else EXCHANGES[0]
primary_root = OKX_DATA if primary == "okx" else BYBIT_DATA
all_coins = list_coins(primary_root)

_crypto_u = filter_crypto_coins(all_coins, crypto=True)
_equity_u = filter_crypto_coins(all_coins, crypto=False)
n_crypto_universe = len(_crypto_u)
n_equity_universe = len(_equity_u)
if ASSET_CLASS == "crypto":
    class_universe = _crypto_u
elif ASSET_CLASS == "equity":
    class_universe = _equity_u
else:
    class_universe = list(all_coins)

if COIN_ALLOWLIST:
    _allow = {str(c).upper().strip() for c in COIN_ALLOWLIST}
    coins = [c for c in class_universe if c in _allow]
elif MAX_COINS and MAX_COINS > 0:
    coins = class_universe[:MAX_COINS]
else:
    coins = list(class_universe)

n_crypto_panel = sum(1 for c in coins if is_crypto(c))
n_equity_panel = len(coins) - n_crypto_panel
print(
    f"asset_class={ASSET_CLASS} universe crypto={n_crypto_universe} equity={n_equity_universe} "
    f"| panel coins={len(coins)} (crypto={n_crypto_panel}, equity={n_equity_panel}) of {len(all_coins)}"
)
print("panel_ts", format_ts(start_ms), "…", format_ts(end_ms))
print("load_from", format_ts(load_min_ms), "(warmup)")


def load_feature_frames(root: Path, coin_list: list[str]) -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    t0 = time.perf_counter()
    for i, coin in enumerate(coin_list):
        bars = load_hist_bars_recent(root, coin, min_ts_ms=load_min_ms, max_ts_ms=end_ms)
        if bars.empty:
            continue
        try:
            frames[coin] = build_ma_ratio_features(
                bars, params=MA_PARAMS, extra_longs=MA_EXTRA_LONGS
            )
        except Exception as exc:  # noqa: BLE001
            print(f"skip {coin}: {exc}")
        if (i + 1) % 25 == 0:
            print(f"  loaded {i+1}/{len(coin_list)} …")
    print(f"loaded {len(frames)} frames in {time.perf_counter()-t0:.1f}s from {root.name}")
    return frames


ex_frames: dict[str, dict[str, pd.DataFrame]] = {}
for ex in EXCHANGES:
    root = OKX_DATA if ex == "okx" else BYBIT_DATA
    ex_coins = [c for c in coins if (root / f"base_coin={c}").is_dir()]
    ex_frames[ex] = load_feature_frames(root, ex_coins)

feature_frames = ex_frames[primary]
print("primary frames", len(feature_frames))


## Панель `(t, coin, composite)` + heatmap

Z heatmap = **raw** канонический composite. Нижняя полоса — средняя по монетам на срезе (интегральный «жар» рынка).  
Color scale: display-only clip `zmax=p99` (панель / Top‑10 не мутируются).


In [ ]:
def panel_timestamps(frames: dict[str, pd.DataFrame], t0: int, t1: int) -> list[int]:
    ts: set[int] = set()
    for fr in frames.values():
        if fr is None or fr.empty:
            continue
        m = (fr["bar_start_ts_ms"] >= t0) & (fr["bar_start_ts_ms"] <= t1)
        ts.update(int(x) for x in fr.loc[m, "bar_start_ts_ms"].tolist())
    return sorted(ts)


def build_panel_for_exchange(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    ts = panel_timestamps(frames, start_ms, end_ms)
    t0 = time.perf_counter()
    panel = build_ma_ratio_panel(
        frames,
        ts,
        variant=COMPOSITE_VARIANT,
        long=MA_LONG,
        primary_long=MA_LONG,
        stride=PANEL_STRIDE,
        store_all_variants=STORE_ALL_VARIANTS,
    )
    print(
        f"panel: n_t={panel['bar_start_ts_ms'].nunique() if not panel.empty else 0} "
        f"n_coin={panel['base_coin'].nunique() if not panel.empty else 0} "
        f"rows={len(panel)} in {time.perf_counter()-t0:.1f}s"
    )
    return panel


panels_ex = {ex: build_panel_for_exchange(fr) for ex, fr in ex_frames.items()}

if len(EXCHANGES) == 1:
    panel = panels_ex[EXCHANGES[0]]
else:
    okx_p = panels_ex["okx"]
    byb_p = panels_ex["bybit"]
    chunks = []
    common_t = sorted(
        set(okx_p["bar_start_ts_ms"]).intersection(byb_p["bar_start_ts_ms"])
    )
    for t in common_t:
        a = okx_p.loc[okx_p["bar_start_ts_ms"] == t]
        b = byb_p.loc[byb_p["bar_start_ts_ms"] == t]
        m = merge_exchange_composites(a, b, how=COMBINE)
        m["bar_start_ts_ms"] = t
        m = m.sort_values("composite", ascending=False, kind="mergesort").reset_index(drop=True)
        m["rank"] = np.arange(1, len(m) + 1)
        chunks.append(m)
    panel = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
    print("merged panel rows", len(panel), "n_t", len(common_t))

assert not panel.empty, "пустая панель — сузьте фильтры или проверьте пути данных"
_preview = [
    "bar_start_ts_ms", "base_coin", "composite", "rank", "rank_xs", "r_vol", "r_atr", "variant",
]
display(panel[[c for c in _preview if c in panel.columns]].head(3))
print(
    "composite finite",
    int(panel["composite"].notna().sum()),
    "/",
    len(panel),
    "range",
    float(panel["composite"].min(skipna=True)),
    "…",
    float(panel["composite"].max(skipna=True)),
)
print("score", SCORE_LABEL)


In [ ]:
EQUITY_SEPARATOR_LABEL = "── equity ──"


def sort_coins_for_heatmap(p: pd.DataFrame, how: str) -> list[str]:
    coins = p["base_coin"].unique().tolist()
    if not coins:
        return []
    if how == "alpha":
        return sorted(coins)
    g = p.groupby("base_coin")["composite"]
    stat = g.max() if how == "max" else g.mean()
    ordered = stat.sort_values(ascending=False).index.tolist()
    missing = [c for c in coins if c not in set(ordered)]
    return ordered + sorted(missing)


def order_coins_with_asset_blocks(
    p: pd.DataFrame, how: str, asset_class: str
) -> tuple[list[str], int, int]:
    coins = p["base_coin"].unique().tolist()
    crypto_coins = [c for c in coins if is_crypto(c)]
    equity_coins = [c for c in coins if not is_crypto(c)]
    n_c, n_e = len(crypto_coins), len(equity_coins)

    if asset_class == "crypto":
        return sort_coins_for_heatmap(p.loc[p["base_coin"].isin(crypto_coins)], how), n_c, n_e
    if asset_class == "equity":
        return sort_coins_for_heatmap(p.loc[p["base_coin"].isin(equity_coins)], how), n_c, n_e

    crypto_order = sort_coins_for_heatmap(p.loc[p["base_coin"].isin(crypto_coins)], how)
    equity_order = sort_coins_for_heatmap(p.loc[p["base_coin"].isin(equity_coins)], how)
    if crypto_order and equity_order:
        return crypto_order + [EQUITY_SEPARATOR_LABEL] + equity_order, n_c, n_e
    return crypto_order + equity_order, n_c, n_e


def _pivot_wide(
    p: pd.DataFrame, value: str, coins_order: list[str], cols: list
) -> np.ndarray:
    real = [c for c in coins_order if c != EQUITY_SEPARATOR_LABEL]
    wide = p.pivot_table(
        index="base_coin", columns="bar_start_ts_ms", values=value, aggfunc="last"
    )
    wide = wide.reindex(index=real, columns=cols)
    arr = wide.to_numpy(dtype=float)
    if EQUITY_SEPARATOR_LABEL not in coins_order:
        return arr
    sep_i = coins_order.index(EQUITY_SEPARATOR_LABEL)
    nan_row = np.full((1, arr.shape[1]), np.nan, dtype=float)
    return np.vstack([arr[:sep_i], nan_row, arr[sep_i:]])


def panel_to_heatmap_arrays(
    p: pd.DataFrame, coins_order: list[str], time_downsample: int
) -> tuple[np.ndarray, list, list[str], np.ndarray, list[str]]:
    real_coins = [c for c in coins_order if c != EQUITY_SEPARATOR_LABEL]
    wide = p.pivot_table(
        index="base_coin", columns="bar_start_ts_ms", values="composite", aggfunc="last"
    )
    wide = wide.reindex(real_coins)
    cols = list(wide.columns)
    if time_downsample > 1:
        cols = cols[::time_downsample]
        wide = wide[cols]
    x_labels = [format_ts(int(c)) for c in cols]
    z_real = wide.to_numpy(dtype=float)
    if EQUITY_SEPARATOR_LABEL in coins_order:
        sep_i = coins_order.index(EQUITY_SEPARATOR_LABEL)
        nan_row = np.full((1, z_real.shape[1]), np.nan, dtype=float)
        z = np.vstack([z_real[:sep_i], nan_row, z_real[sep_i:]])
        y_labels = list(coins_order)
    else:
        z = z_real
        y_labels = list(wide.index)
    hover_cols = [c for c in ("r_vol", "r_atr", "rank_xs") if c in p.columns]
    custom = (
        np.dstack([_pivot_wide(p, c, coins_order, cols) for c in hover_cols])
        if hover_cols
        else np.zeros(z.shape + (0,))
    )
    return z, x_labels, y_labels, custom, hover_cols


coins_order, n_crypto_hm, n_equity_hm = order_coins_with_asset_blocks(
    panel, COIN_SORT, ASSET_CLASS
)
z, x_labels, y_coins, custom, hover_cols = panel_to_heatmap_arrays(
    panel, coins_order, HEATMAP_TIME_DOWNSAMPLE
)
print(
    "heatmap shape (rows × time)",
    z.shape,
    "hover",
    hover_cols,
    f"| n_crypto={n_crypto_hm} n_equity={n_equity_hm} asset_class={ASSET_CLASS}",
)

_z_finite = z[np.isfinite(z)]
if _z_finite.size:
    Z_P99 = float(np.nanpercentile(_z_finite, 99))
    Z_RAW_MAX = float(np.nanmax(_z_finite))
    Z_RAW_MIN = float(np.nanmin(_z_finite))
else:
    Z_P99 = None
    Z_RAW_MAX = None
    Z_RAW_MIN = None
print(
    "heatmap display clip: zmax=p99",
    Z_P99,
    "| raw min/max",
    Z_RAW_MIN,
    "…",
    Z_RAW_MAX,
    "(panel unchanged)",
)

hover_lines = "<br>".join(
    f"{name}=%{{customdata[{i}]:.3f}}" for i, name in enumerate(hover_cols)
)
_p99_title = (
    f" · zmax=p99={Z_P99:.3f} (raw max={Z_RAW_MAX:.3f})"
    if Z_P99 is not None and Z_RAW_MAX is not None
    else ""
)
title_hm = (
    f"Composite heatmap · {SCORE_LABEL} · {START}→{END} · {EXCHANGES} · "
    f"sort={COIN_SORT} · ASSET_CLASS={ASSET_CLASS} · "
    f"n_crypto={n_crypto_hm} n_equity={n_equity_hm}"
    f"{_p99_title}"
)
_heatmap_kwargs = dict(
    z=z,
    x=x_labels,
    y=y_coins,
    colorscale=COLORSCALE,
    colorbar=dict(title="composite (p99 clip)", len=0.72, y=0.62),
    customdata=custom,
    hovertemplate=(
        "coin=%{y}<br>t=%{x}<br>"
        "composite=%{z:.3f}<br>"
        f"{hover_lines}<extra></extra>"
    ),
)
if Z_P99 is not None:
    _heatmap_kwargs["zmax"] = Z_P99

_z_mean_src = np.where(np.isfinite(z), z, np.nan)
_crypto_row_mask = np.array(
    [c != EQUITY_SEPARATOR_LABEL and is_crypto(c) for c in y_coins], dtype=bool
)
_equity_row_mask = np.array(
    [c != EQUITY_SEPARATOR_LABEL and not is_crypto(c) for c in y_coins], dtype=bool
)
with np.errstate(all="ignore"):
    crypto_mean = (
        np.nanmean(_z_mean_src[_crypto_row_mask], axis=0)
        if _crypto_row_mask.any()
        else None
    )
    equity_mean = (
        np.nanmean(_z_mean_src[_equity_row_mask], axis=0)
        if _equity_row_mask.any()
        else None
    )

_mean_title_bits = []
if crypto_mean is not None:
    _mean_title_bits.append("средняя crypto")
if equity_mean is not None:
    _mean_title_bits.append("средняя equity")
MEAN_SERIES_LABEL = " · ".join(_mean_title_bits) if _mean_title_bits else "средние composite"

fig_hm = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.78, 0.22],
    vertical_spacing=0.07,
    subplot_titles=("", MEAN_SERIES_LABEL),
)
fig_hm.add_trace(go.Heatmap(**_heatmap_kwargs), row=1, col=1)
if crypto_mean is not None:
    fig_hm.add_trace(
        go.Scatter(
            x=x_labels,
            y=crypto_mean,
            mode="lines",
            line=dict(width=1.6, color="#1f77b4"),
            name="средняя crypto",
            hovertemplate="t=%{x}<br>средняя crypto=%{y:.4f}<extra></extra>",
        ),
        row=2,
        col=1,
    )
if equity_mean is not None:
    fig_hm.add_trace(
        go.Scatter(
            x=x_labels,
            y=equity_mean,
            mode="lines",
            line=dict(width=1.6, color="#ff7f0e"),
            name="средняя equity",
            hovertemplate="t=%{x}<br>средняя equity=%{y:.4f}<extra></extra>",
        ),
        row=2,
        col=1,
    )
if ASSET_CLASS == "both" and n_crypto_hm and n_equity_hm:
    fig_hm.add_annotation(
        text=f"crypto (n={n_crypto_hm}) ↑ · equity (n={n_equity_hm}) ↓",
        xref="paper",
        yref="paper",
        x=0.0,
        y=1.02,
        showarrow=False,
        font=dict(size=12),
        xanchor="left",
    )
_show_mean_legend = (crypto_mean is not None) and (equity_mean is not None)
fig_hm.update_layout(
    title=title_hm,
    height=max(560, 18 * len(y_coins) + 180),
    margin=dict(l=100, r=40, t=80, b=60),
    showlegend=_show_mean_legend,
    legend=dict(orientation="h", yanchor="bottom", y=0.28, x=0.0),
)
fig_hm.update_xaxes(tickangle=45, nticks=min(24, len(x_labels)), row=1, col=1)
fig_hm.update_xaxes(
    title_text="t (UTC)",
    tickangle=45,
    nticks=min(24, len(x_labels)),
    row=2,
    col=1,
)
fig_hm.update_yaxes(title_text="coin", autorange="reversed", row=1, col=1)
fig_hm.update_yaxes(title_text="composite", row=2, col=1)
fig_hm.show()


## Top‑10 на фиксированном timestamp

Задаёте `TOP10_TS` в CONFIG. На снэпнутом 5m-баре панели — top‑N по **тому же** каноническому `composite`.

Выход: таблица `rank / coin / composite / r_vol / r_atr` и (опционально) detail-графики OHLC+volume+metrics для каждой монеты из Top‑N.


In [ ]:
def parse_ts_ms(value) -> int:
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, float) and np.isfinite(value):
        return int(value)
    s = str(value).strip()
    if s.isdigit():
        return int(s)
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    dt = datetime.fromisoformat(s)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)


def snap_bar_ts(frames_or_panel_ts, t_ms: int) -> int:
    arr = np.asarray(list(frames_or_panel_ts), dtype=np.int64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        raise ValueError("нет timestamps для snap")
    le = arr[arr <= t_ms]
    if le.size:
        return int(le.max())
    return int(arr[np.argmin(np.abs(arr - t_ms))])


def slice_bars_around(frames: dict[str, pd.DataFrame], coin: str, t_ms: int, hours: float) -> pd.DataFrame:
    fr = frames.get(coin)
    if fr is None or fr.empty:
        return pd.DataFrame()
    half = int(hours * 3600 * 1000)
    m = (fr["bar_start_ts_ms"] >= t_ms - half) & (fr["bar_start_ts_ms"] <= t_ms + half)
    return fr.loc[m].copy()


def composite_series(p: pd.DataFrame, coin: str, t0: int, t1: int) -> pd.DataFrame:
    m = (
        (p["base_coin"] == coin)
        & (p["bar_start_ts_ms"] >= t0)
        & (p["bar_start_ts_ms"] <= t1)
    )
    cols = ["bar_start_ts_ms", "composite", "rank", "r_vol", "r_atr", "rank_xs"]
    return p.loc[m, [c for c in cols if c in p.columns]].sort_values("bar_start_ts_ms")


def ensure_coin_features(coin: str) -> pd.DataFrame:
    fr = feature_frames.get(coin)
    if fr is not None and not fr.empty:
        return fr
    print(f"lazy-load features for {coin}")
    bars = load_hist_bars_recent(primary_root, coin, min_ts_ms=load_min_ms, max_ts_ms=end_ms)
    if bars.empty:
        raise ValueError(f"нет баров для {coin} в {primary_root}")
    fr = build_ma_ratio_features(bars, params=MA_PARAMS, extra_longs=MA_EXTRA_LONGS)
    feature_frames[coin] = fr
    return fr


def metrics_from_features(fr: pd.DataFrame, t0: int, t1: int) -> pd.DataFrame:
    m = (fr["bar_start_ts_ms"] >= t0) & (fr["bar_start_ts_ms"] <= t1)
    sub = fr.loc[m].sort_values("bar_start_ts_ms").copy()
    pref = select_composite_column(COMPOSITE_VARIANT, long=MA_LONG, primary_long=MA_LONG)
    if pref in sub.columns:
        sub["composite"] = sub[pref]
    keep = [c for c in ("bar_start_ts_ms", "composite", "r_vol", "r_atr") if c in sub.columns]
    out = sub[keep].copy()
    out["rank"] = np.nan
    return out


def plot_detail_plotly(
    bars: pd.DataFrame,
    comp: pd.DataFrame,
    t_ms: int,
    title: str,
    *,
    score_label: str,
):
    dt = pd.to_datetime(bars["bar_start_ts_ms"], unit="ms", utc=True)
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.5, 0.25, 0.25],
        subplot_titles=("OHLC", "volume", f"metrics · {score_label}"),
    )
    fig.add_trace(
        go.Candlestick(
            x=dt,
            open=bars["open"],
            high=bars["high"],
            low=bars["low"],
            close=bars["close"],
            name="OHLC",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(x=dt, y=bars["volume"], name="volume", marker_color="rgba(100,149,237,0.7)"),
        row=2,
        col=1,
    )
    if not comp.empty:
        cdt = pd.to_datetime(comp["bar_start_ts_ms"], unit="ms", utc=True)
        if "composite" in comp.columns:
            fig.add_trace(
                go.Scatter(x=cdt, y=comp["composite"], name=f"composite ({score_label})", mode="lines"),
                row=3,
                col=1,
            )
        for col, dash in (("r_vol", "dash"), ("r_atr", "dot")):
            if col in comp.columns:
                fig.add_trace(
                    go.Scatter(
                        x=cdt,
                        y=comp[col],
                        name=col,
                        mode="lines",
                        line=dict(dash=dash, width=1),
                        opacity=0.75,
                    ),
                    row=3,
                    col=1,
                )
    t_line = pd.to_datetime(t_ms, unit="ms", utc=True)
    for r in (1, 2, 3):
        fig.add_vline(x=t_line, line_width=2, line_dash="dash", line_color="crimson", row=r, col=1)
    fig.update_layout(
        title=title,
        height=780,
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
        margin=dict(l=60, r=40, t=80, b=40),
    )
    fig.show()
    return fig


assert "panel" in globals() and not panel.empty, "сначала выполните ячейку панели"

top10_ts_req = parse_ts_ms(TOP10_TS)
grid_top = panel["bar_start_ts_ms"].drop_duplicates().sort_values()
top10_ts = snap_bar_ts(grid_top, top10_ts_req)
if top10_ts != top10_ts_req:
    print(f"TOP10_TS snapped {format_ts(top10_ts_req)} → {format_ts(top10_ts)}")

half_ms_top = int(TOP10_WINDOW_HOURS * 3600 * 1000)
at_ts = panel.loc[panel["bar_start_ts_ms"] == top10_ts].copy()
assert not at_ts.empty, f"нет строк панели на {format_ts(top10_ts)}"

_coins = at_ts["base_coin"].astype(str)
if ASSET_CLASS == "crypto":
    at_ts = at_ts.loc[_coins.map(is_crypto)].copy()
elif ASSET_CLASS == "equity":
    at_ts = at_ts.loc[~_coins.map(is_crypto)].copy()

at_ts = at_ts.loc[at_ts["composite"].notna()]
assert not at_ts.empty, (
    f"нет конечных composite на {format_ts(top10_ts)} для ASSET_CLASS={ASSET_CLASS!r}"
)

n_available = int(len(at_ts))
topn_tbl = (
    at_ts.sort_values("composite", ascending=False, kind="mergesort")
    .head(int(TOP_N))
    .reset_index(drop=True)
)
topn_tbl["rank"] = np.arange(1, len(topn_tbl) + 1)

_cols = ["rank", "base_coin", "composite"]
for c in ("r_vol", "r_atr", "rank_xs"):
    if c in topn_tbl.columns:
        _cols.append(c)
_show = topn_tbl[_cols].copy().rename(columns={"base_coin": "coin"})
_show.insert(1, "t_utc", format_ts(top10_ts))

print(
    f"Top‑{len(_show)} по composite @ {format_ts(top10_ts)} · ±{TOP10_WINDOW_HOURS}h · "
    f"ASSET_CLASS={ASSET_CLASS} · {SCORE_LABEL} · "
    f"available={n_available} (requested TOP_N={TOP_N})"
)
if n_available < int(TOP_N):
    print(f"NOTE: на срезе только {n_available} монет с finite composite (< TOP_N={TOP_N})")
display(_show)

figs_topn = []
if PLOT_TOP10_DETAILS:
    need_ohlc = {"open", "high", "low", "close", "volume"}
    for row in topn_tbl.itertuples(index=False):
        coin = str(row.base_coin)
        rank_i = int(row.rank)
        comp_v = float(row.composite)
        fr_c = ensure_coin_features(coin)
        bars_c = slice_bars_around(feature_frames, coin, top10_ts, TOP10_WINDOW_HOURS)
        if bars_c.empty or not need_ohlc.issubset(bars_c.columns):
            bars_c = load_hist_bars_recent(
                primary_root,
                coin,
                min_ts_ms=top10_ts - half_ms_top,
                max_ts_ms=top10_ts + half_ms_top,
            )
        assert not bars_c.empty, f"нет OHLC для {coin} вокруг {format_ts(top10_ts)}"
        comp_c = composite_series(panel, coin, top10_ts - half_ms_top, top10_ts + half_ms_top)
        if comp_c.empty:
            comp_c = metrics_from_features(fr_c, top10_ts - half_ms_top, top10_ts + half_ms_top)
        title_c = (
            f"Top‑{TOP_N} #{rank_i} · {coin} · composite={comp_v:.6g} "
            f"@ {format_ts(top10_ts)} · ±{TOP10_WINDOW_HOURS}h · {SCORE_LABEL}"
        )
        print(title_c)
        figs_topn.append(
            plot_detail_plotly(bars_c, comp_c, top10_ts, title_c, score_label=SCORE_LABEL)
        )
    print("plotted", len(figs_topn), f"detail charts for top‑{TOP_N}")
else:
    print("PLOT_TOP10_DETAILS=False — только таблица")


## Порядок ячеек

1. Imports  
2. CONFIG  
3. Загрузка + features  
4. Панель  
5. Heatmap  
6. Top‑10  

CLI (тот же канон):

```bash
./venv/bin/python research/rank_volatile_coins.py \
  --score-mode ma_ratio --numerator blend --blend-alpha 0.75 --variant geom --long 288 --top 10
```
